# 03 — Sentiment Scoring
Scores each article with VADER (fast) or RoBERTa (accurate).
Input: `data/classified_articles.csv` → Output: `data/scored_articles.csv`

In [1]:
import pandas as pd

df = pd.read_csv('data/classified_articles.csv', parse_dates=['date'])
print(df.shape)

(24584, 9)


## Option A — VADER (recommended for this project)
Fast, no GPU, designed for news/social media. Outputs compound score in [-1, 1].

In [2]:
# pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_score(text: str) -> float:
    # Use headline + first 300 chars of body — VADER degrades on long text
    snippet = str(text)[:300]
    return sia.polarity_scores(snippet)['compound']

df['text_sample'] = df['headline'] + '. ' + df['body'].str[:300]
df['sentiment'] = df['text_sample'].apply(vader_score)

# Map compound to label
df['sentiment_label'] = pd.cut(
    df['sentiment'],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=['negative', 'neutral', 'positive']
)

print(df['sentiment_label'].value_counts())
df[['headline', 'sentiment', 'sentiment_label']].head(5)

sentiment_label
positive    11949
negative    10922
neutral      1713
Name: count, dtype: int64


,headline,sentiment,sentiment_label
0,Rejecting generative AI in healthcare won’t pr...,0.4404,positive
1,Rejecting generative AI in healthcare won’t pr...,0.4404,positive
2,How generative AI in Arc Raiders started a scr...,0.1027,positive
3,How generative AI in Arc Raiders started a scr...,0.1027,positive
4,AI pioneer announces non-profit to develop ‘ho...,0.7906,positive


## Option B — RoBERTa (better accuracy, ~30 min on CPU for 10k)
Use `cardiffnlp/twitter-roberta-base-sentiment-latest` — trained on social/news text.

In [4]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=device,
    truncation=True,
    max_length=512
)

# Batch inference for speed
texts = (df['headline'] + '. ' + df['body'].str[:300]).tolist()
results = sentiment_pipe(texts, batch_size=32)

label_map = {'positive': 1, 'neutral': 0, 'negative': -1}
df['sentiment_label'] = [r['label'].lower() for r in results]
df['sentiment'] = [label_map[r['label'].lower()] * r['score'] for r in results]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [3]:
df.to_csv('data/scored_articles.csv', index=False)
print('Saved: data/scored_articles.csv')
df.describe()

Saved: data/scored_articles.csv


,date,wordcount,sentiment
count,24584,24584.000000,24584.000000
mean,2024-03-24 16:46:18.782948,1562.267369,0.018194
min,2022-01-01 00:00:00,101.000000,-0.985800
25%,2023-05-11 00:00:00,615.000000,-0.542300
50%,2024-04-17 00:00:00,854.000000,0.000000
75%,2025-03-12 00:00:00,1225.000000,0.571900
max,2025-12-31 00:00:00,35484.000000,0.988100
std,NaN,2461.776803,0.600231
